# SciProof Quickstart: Reproducible Scientific Evidence in 2 Minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hayagrivai-labs/sciproof/blob/main/examples/quickstart.ipynb)
[![PyPI](https://img.shields.io/pypi/v/sciproof.svg)](https://pypi.org/project/sciproof/)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.23676138.svg)](https://doi.org/10.5281/zenodo.23676138)

This interactive tutorial demonstrates how to:
1. **Record** computational experiments without modifying code (`sciproof record`)
2. **Calibrate** numerical tolerances across runs (`sciproof calibrate`)
3. **Verify** evidence with cryptographic integrity & rerun comparison (`sciproof verify`)
4. **Export** verified numbers directly as LaTeX macros for papers (`sciproof export --format latex`)

In [ ]:
# 1. Install SciProof (zero runtime dependencies)
!pip install -q sciproof
!sciproof --version

## Step 1: Create a Simulated Experiment Script
We simulate a machine learning or numerical pipeline that outputs `metrics.json`.

In [ ]:
%%writefile train.py
import json, random

# Set seed for repeatability
random.seed(42)

# Simulated experiment metrics
metrics = {
    "accuracy": round(0.942 + random.uniform(-0.0005, 0.0005), 4),
    "loss": round(0.058 + random.uniform(-0.0002, 0.0002), 4),
    "epochs": 20
}

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Experiment complete. Accuracy: {metrics['accuracy']}, Loss: {metrics['loss']}")

## Step 2: Record the Experiment with SciProof
Run `sciproof record` — it runs your command, detects output files, and creates a verified manifest `baseline.sciproof.json`.

In [ ]:
!sciproof record --name baseline -- python train.py

## Step 3: Inspect the Evidence Manifest
The `.sciproof.json` manifest records Python version, OS, package versions, and SHA-256 hashes of all artifacts.

In [ ]:
!sciproof inspect baseline.sciproof.json

## Step 4: Calibrate Floating-Point Noise
`sciproof calibrate` runs your experiment repeatedly and measures variance across runs to suggest realistic tolerances.

In [ ]:
!sciproof calibrate baseline.sciproof.json --runs 3

## Step 5: Verify Evidence (Reviewer Mode)
Re-runs the command in a clean sandbox and checks hashes and tolerances.

In [ ]:
!sciproof verify baseline.sciproof.json --rerun --yes

## Step 6: Export Directly into LaTeX for Papers
Export verified numbers as LaTeX macros — paper values will never drift from code!

In [ ]:
!sciproof export baseline.sciproof.json --format latex -o paper_values.tex \
  --value accuracy metrics.json "$.accuracy" \
  --value loss metrics.json "$.loss"

print("\n--- Generated paper_values.tex ---")
!cat paper_values.tex